# Pocket-aware representation — asli model fix

**Problem jo ye theek karne ki koshish karta hai.** Model target dekh kar rank
nahi karta: 134 checkable targets me se **75 pe** top-50 me ek bhi jaani-pehchani
dawa nahi aati. Wajah ka shaq: ESM-2 poore protein sequence (1000+ residues) ko
ek 1280-number me **mean-pool** karta hai. Us औsat me wo thoda-sa farak, jo ek
kinase ko doosre se alag karta hai — yaani **binding pocket** — doob jaata hai.

**Idea.** Poore sequence ke bajaye **sirf binding pocket** pe ESM chalao. KLIFS
database har human kinase ke liye **85 residue ka pocket** deta hai (structures
se aligned). Us 85-residue ke andar ek residue ka farak = **1.2%** signal,
jabki 1000-residue me wahi farak = **0.1%**. Yaani 12 guna mazboot signal.

**Teen arms, ek hi model/features/splits:**
- **A** = poora sequence ESM (jo abhi hai)
- **B** = sirf pocket ESM
- **C** = dono jod ke (whole + pocket)

**Do cheezein naapenge:**
1. Davis pe CI — random, nayi dawa, naya protein
2. **Mutant discrimination** — ABL1 vs ABL1(E255K) ka representation ab alag
   hota hai ya nahi (asli mutant sequence lagane ke baad)

**Ek faida pehle se:** pocket 85 residue ka hai, to ESM **bahut tez** chalega —
poore sequence se ~12 guna tez.

⛔ **Runtime → T4 GPU** (TPU nahi). Phir **Run all**.

## 1. Tools

In [ ]:
import subprocess, sys
def pip(*a):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a],
                       capture_output=True, text=True)
    print(("✅ " if r.returncode == 0 else "❌ ") + " ".join(a))
    if r.returncode: print(r.stderr[-1200:])
pip("rdkit"); pip("transformers"); pip("xgboost"); pip("scikit-learn")

In [ ]:
import os, json, pickle, re, time, urllib.request, warnings
from collections import OrderedDict, Counter
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr
from sklearn.model_selection import GroupShuffleSplit
RDLogger.DisableLog("rdApp.*")
np.random.seed(0); torch.manual_seed(0)

IS_TPU = bool(os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME"))
if IS_TPU:
    print("⛔ TPU chuna hua hai - T4 GPU chahiye. Runtime → Change runtime type")
GPU = torch.cuda.is_available(); DEVICE = "cuda" if GPU else "cpu"
print("Ready ✅ |", "GPU mila" if GPU else "⚠️ GPU nahi (pocket chhota hai to CPU bhi chal jayega)")

## 2. Settings

In [ ]:
SEEDS      = 5
N_TREES    = 600
ESM_MODEL  = "facebook/esm2_t33_650M_UR50D"
CHEM_MODEL = "DeepChem/ChemBERTa-77M-MTR"
KLIFS_API  = "https://klifs.net/api"

## 3. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/ai_drug_platform"
OUT_DIR = os.path.join(ROOT, "pocket_aware"); os.makedirs(OUT_DIR, exist_ok=True)
print("Drive ready ✅", OUT_DIR)

## 4. KLIFS se pocket laao — **pehle schema dekhte hain, maante nahi**

Aaj teen baar yahi galti ho chuki hai: library ka output "maan" liya aur naam
galat nikla. Isliye ye cell pehle **asli jawab print** karta hai, phir field
dhundhta hai, phir check karta hai ki pocket sach me ~85 residue ka hai.

In [ ]:
def get_json(url, tries=4, timeout=180):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0",
                                                       "Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.load(r)
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
            print(f"  try {k+1}/{tries} fail - {last}")
            time.sleep(3 * (k + 1))
    raise RuntimeError(f"KLIFS se jawab nahi mila: {url} ({last})")

KL_CACHE = os.path.join(OUT_DIR, "klifs_kinases.json")
if os.path.exists(KL_CACHE):
    kin = json.load(open(KL_CACHE))
    print(f"Drive se load: {len(kin)} kinases")
else:
    kin = get_json(f"{KLIFS_API}/kinase_information?species=HUMAN")
    json.dump(kin, open(KL_CACHE, "w"))
    print(f"KLIFS se mile: {len(kin)} human kinases")

assert isinstance(kin, list) and kin, f"Jawab list nahi hai: {type(kin)}"
print("\n--- pehle record ke saare fields (ye MAAN nahi rahe, DEKH rahe hain) ---")
for k, v in kin[0].items():
    sv = str(v)
    print(f"  {k:22s} {sv[:70]}{'...' if len(sv) > 70 else ''}  (len {len(sv)})")

# pocket wala field khud dhundho: string ho, aur 70-110 residue ka ho
AA = set("ACDEFGHIKLMNPQRSTVWY-")
def looks_like_pocket(v):
    if not isinstance(v, str) or not (70 <= len(v) <= 110):
        return False
    up = v.upper()
    return sum(c in AA for c in up) / len(up) > 0.9

cands = [k for k, v in kin[0].items() if looks_like_pocket(v)]
print(f"\npocket jaise fields: {cands}")
assert cands, ("Koi field pocket jaisa nahi laga. Upar ke fields ka screenshot "
               "Claude ko bhejo - field ka naam badla hoga.")
POCKET_FIELD = "pocket" if "pocket" in cands else cands[0]
lens = [len(str(r.get(POCKET_FIELD, ""))) for r in kin if r.get(POCKET_FIELD)]
print(f"chuna field: '{POCKET_FIELD}' · kitne kinases me hai: {len(lens)}/{len(kin)}")
print(f"length: min {min(lens)} max {max(lens)} median {int(np.median(lens))} "
      f"(KLIFS ka pocket 85 hona chahiye)")

# naam ke fields bhi dhundho
NAME_FIELDS = [k for k in kin[0] if k.lower() in
               ("name", "gene_name", "hgnc", "kinase", "full_name", "uniprot")]
print(f"naam wale fields: {NAME_FIELDS}")

## 5. Davis ke proteins ko pocket se jodo
Davis ke naam gene symbol jaise hain (`ABL1`, `EGFR(L858R)`, `JAK2(JH1domain-catalytic)`).
Gene symbol nikaal ke KLIFS ke naam se match karte hain.

In [ ]:
BASES = ["https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data/davis/",
         "https://raw.githubusercontent.com/thinng/GraphDTA/master/data/davis/"]
def davis_file(name, min_bytes=1000):
    p = os.path.join(OUT_DIR, f"davis_{name}")
    if os.path.exists(p) and os.path.getsize(p) >= min_bytes:
        return p
    errs = []
    for b in BASES:
        try:
            req = urllib.request.Request(b + name, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=300) as r, open(p, "wb") as f:
                f.write(r.read())
            if os.path.getsize(p) >= min_bytes:
                return p
        except Exception as e:
            errs.append(str(e))
    raise RuntimeError(f"davis/{name} nahi mila: {errs}")

lig = json.load(open(davis_file("ligands_can.txt")), object_pairs_hook=OrderedDict)
pro = json.load(open(davis_file("proteins.txt")), object_pairs_hook=OrderedDict)
Y = np.asarray(pickle.load(open(davis_file("Y", 10000), "rb"), encoding="latin1"), dtype=float)
Y = -np.log10(Y / 1e9)
SMILES = list(lig.values()); NAMES = list(pro)
SEQ = [str(s).upper().strip() for s in pro.values()]
ii, jj = np.where(~np.isnan(Y))
pairs = pd.DataFrame({"d": ii, "p": jj, "y": Y[ii, jj].astype(np.float32)})
print(f"Davis: {len(SMILES)} dawaiyan x {len(NAMES)} proteins = {len(pairs):,} jode")

def gene_of(n):
    return n.split("(")[0].replace("-nonphosphorylated", "") \
            .replace("-phosphorylated", "").strip().upper()

# KLIFS ka naam -> pocket
k2p = {}
for r in kin:
    pk = str(r.get(POCKET_FIELD, "")).strip()
    if not pk:
        continue
    for f in NAME_FIELDS:
        v = r.get(f)
        if isinstance(v, str) and v.strip():
            k2p.setdefault(v.strip().upper(), pk)

POCKET = [k2p.get(gene_of(n)) for n in NAMES]
got = sum(1 for p in POCKET if p)
print(f"\npocket mila: {got}/{len(NAMES)} Davis proteins ({100*got/len(NAMES):.0f}%)")
miss = sorted({gene_of(n) for n, p in zip(NAMES, POCKET) if not p})
print(f"na mile ({len(miss)} gene): {miss[:25]}{' ...' if len(miss) > 25 else ''}")
assert got >= 0.5 * len(NAMES), (f"Sirf {got} pocket mile - itne kam pe experiment "
                                 f"ka matlab nahi. Upar ka output Claude ko bhejo.")
# jinke pocket nahi, unhe hata dete hain - warna arms ki tulna jaayaz nahi rehti
keep_p = np.array([i for i, p in enumerate(POCKET) if p])
pairs = pairs[pairs.p.isin(set(keep_p.tolist()))].reset_index(drop=True)
remap = {old: new for new, old in enumerate(keep_p)}
pairs["p"] = pairs.p.map(remap)
NAMES = [NAMES[i] for i in keep_p]; SEQ = [SEQ[i] for i in keep_p]
POCKET = [POCKET[i] for i in keep_p]
print(f"\nbache: {len(NAMES)} proteins · {len(pairs):,} jode")
print(f"pocket length: {Counter(len(p) for p in POCKET).most_common(5)}")

## 6. Asli mutant sequences (per-gene offset wala tareeka)
Davis har mutant ko wild-type ki **hubahu** sequence deta hai. Ye pehle naapa ja
chuka hai. Yahan wahi fix lagate hain, taaki mutant discrimination test ka koi
matlab bane.

In [ ]:
POINT = re.compile(r"^([A-Z])(\d+)([A-Z])$")
DEL = re.compile(r"^([A-Z])(\d+)([A-Z]?)(\d+)del$")
MULTI = re.compile(r"(?=([A-Z]\d+[A-Z]))")
AA20 = set("ACDEFGHIKLMNPQRSTVWY")

def split_edits(inside):
    inside = inside.strip()
    if not inside or inside.lower() in ("itd", "jh1domain-catalytic",
                                        "jh2domain-pseudokinase"):
        return []
    out = []
    for p in re.split(r"[,\-]", inside):
        p = p.strip()
        if not p: continue
        if DEL.match(p) or POINT.match(p): out.append(p); continue
        found = [m.group(1) for m in MULTI.finditer(p)]
        out.extend(found if found else [])
    return out

def gene_offset(seq, edits, span=60):
    pts = []
    for e in edits:
        m = POINT.match(e) or DEL.match(e)
        if m: pts.append((m.group(1), int(m.group(2))))
    if not pts: return 0, 0, 0
    best, bn = 0, -1
    for off in sorted(range(-span, span + 1), key=lambda o: (abs(o), o)):
        n = sum(1 for aa, pos in pts
                if 0 <= pos - 1 + off < len(seq) and seq[pos - 1 + off] == aa)
        if n > bn: best, bn = off, n
    return best, bn, len(pts)

per_gene = {}
for n, s in zip(NAMES, SEQ):
    g = gene_of(n)
    m = re.match(r"^([^(]+)\((.*)\)(p?)$", n)
    ed = split_edits(m.group(2)) if m else []
    d = per_gene.setdefault(g, {"edits": [], "seq": s})
    d["edits"].extend(ed)
OFF = {g: gene_offset(d["seq"], d["edits"])[0] for g, d in per_gene.items()}

def apply_mut(name, seq):
    m = re.match(r"^([^(]+)\((.*)\)(p?)$", name)
    if not m: return seq, "wt"
    edits = split_edits(m.group(2))
    if not edits: return seq, "skip"
    off = OFF.get(gene_of(name), 0)
    s = list(seq); done = fail = 0
    for e in sorted(edits, key=lambda x: x.endswith("del")):
        pm = POINT.match(e)
        if pm:
            frm, pos, to = pm.group(1), int(pm.group(2)), pm.group(3)
            i = pos - 1 + off
            if to in AA20 and 0 <= i < len(s) and s[i] == frm:
                s[i] = to; done += 1
            else: fail += 1
            continue
        dm = DEL.match(e)
        if dm:
            frm, a, _, b = dm.group(1), int(dm.group(2)), dm.group(3), int(dm.group(4))
            i = a - 1 + off
            if 0 <= i < len(s) and s[i] == frm and b >= a and i + (b - a + 1) <= len(s):
                del s[i:i + (b - a + 1)]; done += 1
            else: fail += 1
            continue
        fail += 1
    if done and not fail: return "".join(s), "ok"
    if done: return "".join(s), "partial"
    return seq, "skip"

SEQ_MUT, st = [], []
for n, s in zip(NAMES, SEQ):
    a, b = apply_mut(n, s); SEQ_MUT.append(a); st.append(b)
print("mutation status:", dict(Counter(st)))
print(f"unique sequences: {len(set(SEQ))} -> {len(set(SEQ_MUT))}")

# Pocket pe bhi wahi mutation lagana hai, warna arm B me mutants same reh jayenge.
# Pocket aligned 85-residue string hai, usme Davis ki numbering nahi chalti -
# isliye pocket me wo residue badalte hain jo WT sequence me badla, agar wo
# residue pocket me maujood ho (position se nahi, context se).
def mutate_pocket(pk, wt, mut):
    diffs = [(i, wt[i], mut[i]) for i in range(min(len(wt), len(mut))) if wt[i] != mut[i]]
    if not diffs or len(diffs) > 4:
        return pk, 0
    out, changed = list(pk), 0
    for _, frm, to in diffs:
        # pocket me us residue ka ek hi occurrence ho to hi badlo - warna galat
        # jagah badalne ka khatra hai (aaj wahi galti pakdi ja chuki hai)
        idx = [k for k, c in enumerate(out) if c == frm]
        if len(idx) == 1:
            out[idx[0]] = to; changed += 1
    return "".join(out), changed

POCKET_MUT, nch = [], 0
for pk, wt, mu in zip(POCKET, SEQ, SEQ_MUT):
    p2, c = mutate_pocket(pk, wt, mu); POCKET_MUT.append(p2); nch += (c > 0)
print(f"pocket me mutation laga: {nch} proteins")
print(f"unique pockets: {len(set(POCKET))} -> {len(set(POCKET_MUT))}")

## 7. Features — teen arms ke liye
Molecule features ek hi baar. Protein ke liye **teen** version: poora sequence,
sirf pocket, aur dono jod ke.

In [ ]:
fp1024 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
DESC = Descriptors._descList
_MDL = {}
def _get(name, half=False):
    if name not in _MDL:
        tok = AutoTokenizer.from_pretrained(name)
        mdl = AutoModel.from_pretrained(name).to(DEVICE).eval()
        if half and GPU: mdl = mdl.half()
        _MDL[name] = (tok, mdl)
    return _MDL[name]

def drug_feats(smis):
    out = []
    for s in smis:
        m = Chem.MolFromSmiles(str(s))
        d = []
        for _, fn in DESC:
            try: d.append(float(fn(m)))
            except Exception: d.append(np.nan)
        d = np.nan_to_num(np.clip(np.array(d, np.float64), -1e6, 1e6)).astype(np.float32)
        out.append(np.concatenate([fp1024.GetFingerprintAsNumPy(m).astype(np.float32), d]))
    return np.stack(out)

@torch.no_grad()
def chemberta(smis, batch=128):
    tok, mdl = _get(CHEM_MODEL); out = []
    for i in range(0, len(smis), batch):
        t = tok([str(x) for x in smis[i:i+batch]], return_tensors="pt", padding=True,
                truncation=True, max_length=256).to(DEVICE)
        h = mdl(**t).last_hidden_state
        m = t["attention_mask"].unsqueeze(-1).float()
        out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

@torch.no_grad()
def esm(seqs, window=1000):
    tok, mdl = _get(ESM_MODEL, half=True); out = []
    for k, s in enumerate(seqs):
        s = str(s).upper().replace("-", "")       # pocket me gaps ho sakte hain
        chunks = [s[i:i+window] for i in range(0, len(s), window)] or [s]
        vs = []
        for c in chunks:
            t = tok(c, return_tensors="pt").to(DEVICE)
            vs.append(mdl(**t).last_hidden_state[0, 1:-1].float().mean(0).cpu().numpy())
        out.append(np.mean(vs, 0))
        if (k + 1) % 100 == 0: print(f"    ESM {k+1}/{len(seqs)}", end="\r")
    return np.stack(out).astype(np.float32)

def cached(name, fn):
    p = os.path.join(OUT_DIR, name + ".npy")
    if os.path.exists(p):
        a = np.load(p); print(f"  Drive se load: {name} {a.shape}"); return a
    t0 = time.time(); a = fn(); np.save(p, a)
    print(f"  bana: {name} {a.shape} ({(time.time()-t0)/60:.1f} min)"); return a

DRUG_X = np.hstack([cached("drug_rdkit", lambda: drug_feats(SMILES)),
                    cached("drug_chem", lambda: chemberta(SMILES))]).astype(np.float32)
E_WHOLE  = cached("esm_whole_mut", lambda: esm(SEQ_MUT))
E_POCKET = cached("esm_pocket_mut", lambda: esm(POCKET_MUT))
for _k in list(_MDL): del _MDL[_k]
if GPU: torch.cuda.empty_cache()
print(f"\nDrug {DRUG_X.shape} · whole {E_WHOLE.shape} · pocket {E_POCKET.shape}")

## 8. Mutant discrimination — pocket se farak padta hai kya?
Pehle naapa gaya tha: poore sequence pe ABL1 aur ABL1(E255K) ke embeddings
**bit-identical** the (kyunki Davis ki sequence hi same thi). Asli sequence
lagane ke baad ab dekhte hain ki **kaun sa arm** unko behtar alag karta hai.

In [ ]:
def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

base_of = lambda n: n.split("(")[0]
groups = {}
for i, n in enumerate(NAMES):
    groups.setdefault(base_of(n), []).append(i)

rows = []
for g, idx in groups.items():
    wts = [i for i in idx if "(" not in NAMES[i]]
    muts = [i for i in idx if re.search(r"\([A-Z]\d+[A-Z]", NAMES[i])]
    if not wts or not muts: continue
    w = wts[0]
    for m in muts:
        rows.append({"gene": g, "mutant": NAMES[m],
                     "whole_cos": cos(E_WHOLE[w], E_WHOLE[m]),
                     "pocket_cos": cos(E_POCKET[w], E_POCKET[m]),
                     "seq_same": SEQ_MUT[w] == SEQ_MUT[m],
                     "pocket_same": POCKET_MUT[w] == POCKET_MUT[m]})
mt = pd.DataFrame(rows)
if len(mt):
    print(f"{len(mt)} wild-type/mutant jode\n")
    print(mt.head(12).to_string(index=False))
    print(f"\n1.000000 = bilkul same vector (yaani farak hi nahi)")
    print(f"  poora sequence: {int((mt.whole_cos > 0.999999).sum())}/{len(mt)} jode bilkul same")
    print(f"  sirf pocket   : {int((mt.pocket_cos > 0.999999).sum())}/{len(mt)} jode bilkul same")
    print(f"\n  average cosine - whole {mt.whole_cos.mean():.6f} · pocket {mt.pocket_cos.mean():.6f}")
    print("  (kam cosine = zyada farak = behtar discrimination)")
else:
    print("Koi wild-type/mutant jodi nahi mili")

## 9. Teen arms ka muqabla — **iska screenshot Claude ko bhejna hai**
Model, features, hyperparameters, splits — sab same. Sirf protein representation badli.

In [ ]:
import xgboost as xgb

def ci(y_true, y_pred, chunk=2000):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    num = den = 0.0
    for s in range(0, len(y_true), chunk):
        yt, yp = y_true[s:s+chunk, None], y_pred[s:s+chunk, None]
        mask = yt > y_true[None, :]
        diff = yp - y_pred[None, :]
        num += ((diff > 0) & mask).sum() + 0.5 * ((diff == 0) & mask).sum()
        den += mask.sum()
    return num / den if den else float("nan")

ARMS = {"A whole": E_WHOLE, "B pocket": E_POCKET,
        "C whole+pocket": np.hstack([E_WHOLE, E_POCKET]).astype(np.float32)}
res = []
for seed in range(SEEDS):
    for split in ("random", "nayi dawa", "naya protein"):
        if split == "random":
            rs = np.random.default_rng(seed); perm = rs.permutation(len(pairs))
            cut = int(0.8 * len(pairs)); tr, te = perm[:cut], perm[cut:]
        else:
            grp = pairs.d if split == "nayi dawa" else pairs.p
            tr, te = next(GroupShuffleSplit(1, test_size=0.2,
                                            random_state=seed).split(pairs, groups=grp))
        trp, tep = pairs.iloc[tr], pairs.iloc[te]
        for arm, E in ARMS.items():
            Xtr = np.hstack([DRUG_X[trp.d.values], E[trp.p.values]]).astype(np.float32)
            Xte = np.hstack([DRUG_X[tep.d.values], E[tep.p.values]]).astype(np.float32)
            m = xgb.XGBRegressor(n_estimators=N_TREES, max_depth=8, learning_rate=0.06,
                                 subsample=0.8, colsample_bytree=0.6, tree_method="hist",
                                 device=DEVICE, random_state=0)
            m.fit(Xtr, trp.y.values); p = m.predict(Xte)
            res.append({"seed": seed, "split": split, "arm": arm,
                        "CI": float(ci(tep.y.values, p)),
                        "MSE": float(np.mean((tep.y.values - p) ** 2)),
                        "Pearson": float(pearsonr(tep.y.values, p)[0])})
            del Xtr, Xte
    print(f"  seed {seed} done")
    pd.DataFrame(res).to_csv(os.path.join(OUT_DIR, "per_seed.csv"), index=False)

df = pd.DataFrame(res)
print("\n=== CI (zyada = behtar) ===")
piv = df.pivot_table(index="split", columns="arm", values="CI").round(4)
print(piv.to_string())
print("\n=== A se farak (t = mean/SE, |t|>=2 = shor se bada) ===")
out = []
for split in df.split.unique():
    s = df[df.split == split].pivot(index="seed", columns="arm", values="CI")
    for arm in ("B pocket", "C whole+pocket"):
        d = (s[arm] - s["A whole"]).dropna()
        se = d.std() / np.sqrt(len(d)) if len(d) > 1 else float("nan")
        out.append({"split": split, "arm": arm, "delta_CI": round(d.mean(), 4),
                    "jeete": f"{int((d>0).sum())}/{len(d)}",
                    "t": round(d.mean()/se, 2) if se else None})
cmp_df = pd.DataFrame(out)
print(cmp_df.to_string(index=False))

best = cmp_df.sort_values("delta_CI", ascending=False).iloc[0]
solid = best["t"] is not None and best["t"] >= 2 and best.delta_CI > 0
verdict = (f"✅ POCKET SE FAIDA - '{best.arm}' ne {best.split} pe CI {best.delta_CI:+.4f} "
           f"(t={best['t']}, {best.jeete}). Ye raasta sahi hai."
           if solid else
           f"❌ pocket se pakka faida nahi - sabse achha '{best.arm}' bhi "
           f"{best.split} pe CI {best.delta_CI:+.4f} (t={best['t']}, {best.jeete}), "
           f"shor ke andar.")
print(f"\n>>> {verdict}")

json.dump({"verdict": verdict, "n_proteins": len(NAMES), "n_pairs": int(len(pairs)),
           "pocket_coverage": f"{got}/{got + len(miss)}",
           "ci": piv.to_dict(), "compare": out,
           "mutant_whole_identical": int((mt.whole_cos > 0.999999).sum()) if len(mt) else None,
           "mutant_pocket_identical": int((mt.pocket_cos > 0.999999).sum()) if len(mt) else None},
          open(os.path.join(OUT_DIR, "pocket_result.json"), "w"), indent=2)
print(f"\nsave: {OUT_DIR}/pocket_result.json")

**Kaise padhein.** Sabse zaroori row **naya protein** (cold-target) hai — wahi
batati hai ki model naye kinase pe kaam karega ya nahi, aur repurposing wahi pe
fail ho raha hai. Agar wahan pocket jeetta hai, to 75 fail targets pe asli umeed
hai.

Section 8 alag se batata hai ki pocket mutants ko behtar alag karta hai ya nahi —
ye ek swatantra sabooot hai, CI se alag.